# Profile Orderflow Strategy v15
## Purpose
v15 is a research-grade rewrite of v14. It preserves the valid auction-market thesis while correcting the implementation, execution, and reporting defects identified during review.

## Core thesis
* **Location first:** use prior-day / rolling-24h Volume Profile and a rolling five-completed-day composite.
* **Auction state second:** distinguish overlapping balance, higher-value acceptance, lower-value acceptance, failed lower auction, and failed upper auction.
* **Orderflow timing third:** require evidence from three distinct categories—aggression, effectiveness, and acceptance—rather than correlated 3-of-5 votes.
* **Execution last:** signal on a completed 5-minute bar, fill on the next available trade/quote or next bar open, simulate stop/target touches with intrabar high/low and conservative same-bar ordering.

## Important scope
* `VAL_REJECTION_LONG` is the only active production-research setup by default.
* `VAL_REJECTION_SHORT` and `VAH_RECLAIM_LONG` are diagnostics-only by default.
* Binance bookTicker is treated as sampled L1 displayed-liquidity context, not footprint or stacked imbalance.
* Exact-price absorption is not claimed unless price-level trade data support the calculation.

## NautilusTrader integration
The notebook initializes the Binance BTCUSDT perpetual test instrument through `TestInstrumentProvider` when NautilusTrader is installed. The research engine below remains explicit and auditable because the custom auction/profile state, candidate funnels, and partial-position logic are domain-specific. The output can subsequently be converted to Nautilus `TradeTick`, `QuoteTick`, and order events for a full `BacktestEngine`/`BacktestNode` replay.

In [ ]:
from __future__ import annotations

import io
import json
import math
import os
import warnings
from collections import defaultdict, deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd

try:
    from nautilus_trader.test_kit.providers import TestInstrumentProvider
    NAUTILUS_AVAILABLE = True
    INSTRUMENT = TestInstrumentProvider.btcusdt_perp_binance()
except ImportError:
    NAUTILUS_AVAILABLE = False
    INSTRUMENT = None

warnings.filterwarnings("ignore")

SYMBOL = "BTCUSDT"
TICK_SIZE = 0.10
VALUE_AREA_PCT = 0.70
BAR_MINUTES = 5
BAR_MS = BAR_MINUTES * 60_000
BARS_24H = 1440 // BAR_MINUTES      # 12 bars
BARS_60M = 60 // BAR_MINUTES        # 36 bars
BARS_180M = 180 // BAR_MINUTES      # 288 bars

ACCOUNT_START = 100_000.0
BASE_RISK_LONG = 0.010
BASE_RISK_SHORT = 0.0050
MAX_LEVERAGE = 3.0
MAX_BTC = 5.0
DAILY_LOSS_LIMIT = 0.02

TAKER_FEE_BPS = 5.0
SLIPPAGE_BPS = 1.0
SPREAD_LIMIT_BPS = 10.0

OUT_DIR = Path("outputs/v15")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"NautilusTrader available: {NAUTILUS_AVAILABLE}")
if INSTRUMENT is not None:
    print(f"Instrument: {INSTRUMENT.id}")

In [ ]:
@dataclass(frozen=True)
class VolumeProfile:
    val: float
    poc: float
    vah: float
    total_volume: float
    price_volume: dict[float, float]

    @property
    def midpoint(self) -> float:
        return (self.vah + self.val) / 2.0

    @property
    def span(self) -> float:
        return max(self.vah - self.val, TICK_SIZE)


@dataclass
class Bar:
    ts_start: int
    ts_end: int
    open: float
    high: float
    low: float
    close: float
    volume: float
    buy_volume: float
    sell_volume: float
    delta: float
    delta_pct: float
    buy_qty_sum: float
    sell_qty_sum: float
    bid_qty_sum: float
    ask_qty_sum: float
    spread_bps_median: float
    volume_by_price: dict[float, float] = field(default_factory=dict)

    @property
    def close_location(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.5
        return (self.close - self.low) / span

    @property
    def body_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return abs(self.close - self.open) / span

    @property
    def lower_wick_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return (min(self.open, self.close) - self.low) / span

    @property
    def upper_wick_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return (self.high - max(self.open, self.close)) / span

    @property
    def msg_imbalance(self) -> float:
        # Positive = more displayed bid size; negative = more displayed ask size.
        return math.log((self.bid_qty_sum + 1e-9) / (self.ask_qty_sum + 1e-9))


def bucket_price(px: float) -> float:
    return round(round(px / TICK_SIZE) * TICK_SIZE, 8)


def compute_volume_profile(price_volume: dict[float, float], value_area_pct: float = VALUE_AREA_PCT) -> Optional[VolumeProfile]:
    if not price_volume:
        return None
    pv = {float(k): float(v) for k, v in price_volume.items() if v > 0.0}
    if not pv:
        return None
    prices = sorted(pv.keys())
    poc = max(prices, key=lambda p: (pv[p], -abs(p - np.average(prices, weights=[pv[x] for x in prices]))))
    total = sum(pv.values())
    target = total * value_area_pct
    selected = {poc}
    running = pv[poc]
    left_idx = prices.index(poc) - 1
    right_idx = prices.index(poc) + 1
    
    while running < target and (left_idx >= 0 or right_idx < len(prices)):
        lv = pv[prices[left_idx]] if left_idx >= 0 else -1.0
        rv = pv[prices[right_idx]] if right_idx < len(prices) else -1.0
        if lv >= rv and left_idx >= 0:
            selected.add(prices[left_idx])
            running += lv
            left_idx -= 1
        elif right_idx < len(prices):
            selected.add(prices[right_idx])
            running += rv
            right_idx += 1
        else:
            if left_idx >= 0:
                selected.add(prices[left_idx])
                running += lv
                left_idx -= 1
            else:
                break
    return VolumeProfile(min(selected), poc, max(selected), total, pv)


def merge_profiles(profiles: Iterable[VolumeProfile]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for prof in profiles:
        for px, qty in prof.price_volume.items():
            merged[px] += qty
    return compute_volume_profile(merged)


def profile_from_bars(bars: list[Bar]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for b in bars:
        for px, qty in b.volume_by_price.items():
            merged[px] += qty
    return compute_volume_profile(merged)


def build_5m_bars(trades: pd.DataFrame, book_ticker: Optional[pd.DataFrame] = None) -> list[Bar]:
    """
    Build completed 5-minute bars without future leakage.
    """
    # Required aggtrades columns: transact_time, price, quantity, is_buyer_maker.
    # 'is_buyer_maker' means the buyer was passive, so the aggressive side was sell.
    required = {"transact_time", "price", "quantity", "is_buyer_maker"}
    missing = required - set(trades.columns)
    if missing:
        raise ValueError(f"Missing aggtrades columns: {sorted(missing)}")
        
    t = trades.copy()
    t = t.sort_values("transact_time", kind="stable")
    t["bar_start"] = (t["transact_time"].astype("int64") // BAR_MS) * BAR_MS
    t["agg_buy_qty"] = np.where(t["is_buyer_maker"] == False, t["quantity"], 0.0)
    t["agg_sell_qty"] = np.where(t["is_buyer_maker"] == True, t["quantity"], 0.0)
    t["bucket"] = t["price"].map(bucket_price)
    
    quote_groups: dict[int, tuple[float, float, float]] = {}
    if book_ticker is not None and not book_ticker.empty:
        q = book_ticker.copy()
        q = q.sort_values("transaction_time", kind="stable")
        q["bar_start"] = (q["transaction_time"].astype("int64") // BAR_MS) * BAR_MS
        q["spread_bps"] = ((q["best_ask_price"] - q["best_bid_price"]) / ((q["best_ask_price"] + q["best_bid_price"]) / 2.0)) * 10_000
        quote_groups = (
            q.groupby("bar_start")
            .apply(lambda df: (
                float(df["best_bid_qty"].sum()),
                float(df["best_ask_qty"].sum()),
                float(df["spread_bps"].median())
            ))
            .to_dict()
        )
        
    out_list: list[Bar] = []
    for ts, g in t.groupby("bar_start", sort=True):
        p_vals = g["price"].values
        q_vals = g["quantity"].values
        sell_f = float(g["agg_sell_qty"].sum())
        buy_f = float(g["agg_buy_qty"].sum())
        vol = buy_f + sell_f
        
        pv = g.groupby("bucket", sort=False)["quantity"].sum().astype(float).to_dict()
        bid_q, ask_q, spread_median = quote_groups.get(ts, (0.0, 0.0, 0.0))
        
        out_list.append(Bar(
            ts_start=int(ts),
            ts_end=int(ts + BAR_MS),
            open=float(p_vals[0]),
            high=float(p_vals.max()),
            low=float(p_vals.min()),
            close=float(p_vals[-1]),
            volume=vol,
            buy_volume=buy_f,
            sell_volume=sell_f,
            delta=buy_f - sell_f,
            delta_pct=(buy_f - sell_f) / vol if vol > 0.0 else 0.0,
            buy_qty_sum=buy_f,
            sell_qty_sum=sell_f,
            bid_qty_sum=bid_q,
            ask_qty_sum=ask_q,
            spread_bps_median=spread_median,
            volume_by_price=pv
        ))
    return out_list


def assert_bars_valid(bars: list[Bar]) -> bool:
    if not bars:
        return True
    last_ts = bars[0].ts_start
    for b in bars:
        assert b.ts_start >= last_ts, "Bars must be strictly timestamp ordered"
        assert b.low <= min(b.open, b.close) <= max(b.open, b.close) <= b.high
        assert abs(b.buy_volume + b.sell_volume - b.volume) <= 1e-6
        assert abs((b.buy_volume - b.sell_volume) - b.delta) <= 1e-6
        assert -1.0 <= b.delta_pct <= 1.0
        last_ts = b.ts_start
    return True

In [ ]:
@dataclass(frozen=True)
class VolumeProfile:
    val: float
    poc: float
    vah: float
    total_volume: float
    price_volume: dict[float, float]

    @property
    def midpoint(self) -> float:
        return (self.vah + self.val) / 2.0

    @property
    def span(self) -> float:
        return max(self.vah - self.val, TICK_SIZE)


@dataclass
class Bar:
    ts_start: int
    ts_end: int
    open: float
    high: float
    low: float
    close: float
    volume: float
    buy_volume: float
    sell_volume: float
    delta: float
    delta_pct: float
    buy_qty_sum: float
    sell_qty_sum: float
    bid_qty_sum: float
    ask_qty_sum: float
    spread_bps_median: float
    volume_by_price: dict[float, float] = field(default_factory=dict)

    @property
    def close_location(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.5
        return (self.close - self.low) / span

    @property
    def body_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return abs(self.close - self.open) / span

    @property
    def lower_wick_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return (min(self.open, self.close) - self.low) / span

    @property
    def upper_wick_ratio(self) -> float:
        span = self.high - self.low
        if span <= 0.0:
            return 0.0
        return (self.high - max(self.open, self.close)) / span

    @property
    def msg_imbalance(self) -> float:
        # Positive = more displayed bid size; negative = more displayed ask size.
        return math.log((self.bid_qty_sum + 1e-9) / (self.ask_qty_sum + 1e-9))


def bucket_price(px: float) -> float:
    return round(round(px / TICK_SIZE) * TICK_SIZE, 8)


def compute_volume_profile(price_volume: dict[float, float], value_area_pct: float = VALUE_AREA_PCT) -> Optional[VolumeProfile]:
    if not price_volume:
        return None
    pv = {float(k): float(v) for k, v in price_volume.items() if v > 0.0}
    if not pv:
        return None
    prices = sorted(pv.keys())
    poc = max(prices, key=lambda p: (pv[p], -abs(p - np.average(prices, weights=[pv[x] for x in prices]))))
    total = sum(pv.values())
    target = total * value_area_pct
    selected = {poc}
    running = pv[poc]
    left_idx = prices.index(poc) - 1
    right_idx = prices.index(poc) + 1
    
    while running < target and (left_idx >= 0 or right_idx < len(prices)):
        lv = pv[prices[left_idx]] if left_idx >= 0 else -1.0
        rv = pv[prices[right_idx]] if right_idx < len(prices) else -1.0
        if lv >= rv and left_idx >= 0:
            selected.add(prices[left_idx])
            running += lv
            left_idx -= 1
        elif right_idx < len(prices):
            selected.add(prices[right_idx])
            running += rv
            right_idx += 1
        else:
            if left_idx >= 0:
                selected.add(prices[left_idx])
                running += lv
                left_idx -= 1
            else:
                break
    return VolumeProfile(min(selected), poc, max(selected), total, pv)


def merge_profiles(profiles: Iterable[VolumeProfile]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for prof in profiles:
        for px, qty in prof.price_volume.items():
            merged[px] += qty
    return compute_volume_profile(merged)


def profile_from_bars(bars: list[Bar]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for b in bars:
        for px, qty in b.volume_by_price.items():
            merged[px] += qty
    return compute_volume_profile(merged)


def build_5m_bars(trades: pd.DataFrame, book_ticker: Optional[pd.DataFrame] = None) -> list[Bar]:
    """
    Build completed 5-minute bars without future leakage.
    """
    # Required aggtrades columns: transact_time, price, quantity, is_buyer_maker.
    # 'is_buyer_maker' means the buyer was passive, so the aggressive side was sell.
    required = {"transact_time", "price", "quantity", "is_buyer_maker"}
    missing = required - set(trades.columns)
    if missing:
        raise ValueError(f"Missing aggtrades columns: {sorted(missing)}")
        
    t = trades.copy()
    t = t.sort_values("transact_time", kind="stable")
    t["bar_start"] = (t["transact_time"].astype("int64") // BAR_MS) * BAR_MS
    t["agg_buy_qty"] = np.where(t["is_buyer_maker"] == False, t["quantity"], 0.0)
    t["agg_sell_qty"] = np.where(t["is_buyer_maker"] == True, t["quantity"], 0.0)
    t["bucket"] = t["price"].map(bucket_price)
    
    quote_groups: dict[int, tuple[float, float, float]] = {}
    if book_ticker is not None and not book_ticker.empty:
        q = book_ticker.copy()
        q = q.sort_values("transaction_time", kind="stable")
        q["bar_start"] = (q["transaction_time"].astype("int64") // BAR_MS) * BAR_MS
        q["spread_bps"] = ((q["best_ask_price"] - q["best_bid_price"]) / ((q["best_ask_price"] + q["best_bid_price"]) / 2.0)) * 10_000
        quote_groups = (
            q.groupby("bar_start")
            .apply(lambda df: (
                float(df["best_bid_qty"].sum()),
                float(df["best_ask_qty"].sum()),
                float(df["spread_bps"].median())
            ))
            .to_dict()
        )
        
    out_list: list[Bar] = []
    for ts, g in t.groupby("bar_start", sort=True):
        p_vals = g["price"].values
        q_vals = g["quantity"].values
        sell_f = float(g["agg_sell_qty"].sum())
        buy_f = float(g["agg_buy_qty"].sum())
        vol = buy_f + sell_f
        
        pv = g.groupby("bucket", sort=False)["quantity"].sum().astype(float).to_dict()
        bid_q, ask_q, spread_median = quote_groups.get(ts, (0.0, 0.0, 0.0))
        
        out_list.append(Bar(
            ts_start=int(ts),
            ts_end=int(ts + BAR_MS),
            open=float(p_vals[0]),
            high=float(p_vals.max()),
            low=float(p_vals.min()),
            close=float(p_vals[-1]),
            volume=vol,
            buy_volume=buy_f,
            sell_volume=sell_f,
            delta=buy_f - sell_f,
            delta_pct=(buy_f - sell_f) / vol if vol > 0.0 else 0.0,
            buy_qty_sum=buy_f,
            sell_qty_sum=sell_f,
            bid_qty_sum=bid_q,
            ask_qty_sum=ask_q,
            spread_bps_median=spread_median,
            volume_by_price=pv
        ))
    return out_list


def assert_bars_valid(bars: list[Bar]) -> bool:
    if not bars:
        return True
    last_ts = bars[0].ts_start
    for b in bars:
        assert b.ts_start >= last_ts, "Bars must be strictly timestamp ordered"
        assert b.low <= min(b.open, b.close) <= max(b.open, b.close) <= b.high
        assert abs(b.buy_volume + b.sell_volume - b.volume) <= 1e-6
        assert abs((b.buy_volume - b.sell_volume) - b.delta) <= 1e-6
        assert -1.0 <= b.delta_pct <= 1.0
        last_ts = b.ts_start
    return True

## Auction context model
v15 removes the six-vote composite score. Context is represented as interpretable geometry and acceptance:
* **Composite geometry:** rolling five completed days only; no current-day or future data.
* **Intraday geometry:** true rolling 60-minute (12 bars) and 180-minute (36 bars) profiles.
* **Material migration:** profile shifts are normalized by active value-area width and ATR, and must persist.
* **Acceptance:** uses both close counts and the fraction of volume transacted outside the reference boundary.

CVD is deliberately excluded from the higher-timeframe context and reserved for entry orderflow timing.

In [ ]:
@dataclass(frozen=True)
class ContextSnapshot:
    geometry: str
    migration: str
    acceptance: str
    p60: Optional[VolumeProfile]
    p180: Optional[VolumeProfile]
    poc_shift_60_norm: float
    midpoint_shift_60_norm: float
    overlap_60: float


def value_overlap(a: VolumeProfile, b: VolumeProfile) -> float:
    overlap = max(0.0, min(a.vah, b.vah) - max(a.val, b.val))
    denom = max(a.vah - a.val, b.vah - b.val)
    denom = max(denom, TICK_SIZE)
    return min(overlap / denom, 1.0)


class AuctionContextEngine:
    def __init__(self, persistence: int = 2):
        self.persistence = persistence
        self.bars: list[Bar] = []
        self.self_prev_p60: Optional[VolumeProfile] = None
        self.self_prev_p180: Optional[VolumeProfile] = None
        self.self_persistence = persistence
        self.raw_migration_history: deque[str] = deque(maxlen=persistence)

    def update(self, bar: Bar, active: VolumeProfile, composite: VolumeProfile, atr: float) -> ContextSnapshot:
        self.bars.append(bar)
        
        # P60 from recent 12 bars, P180 from recent 36 bars
        p60 = profile_from_bars(self.bars[-BARS_60M:]) if len(self.bars) >= BARS_60M else None
        p180 = profile_from_bars(self.bars[-BARS_180M:]) if len(self.bars) >= BARS_180M else None
        
        geometry = "OVERLAPPING_COMPOSITE"
        if active.val < composite.val and active.vah > composite.vah:
            geometry = "OUTSIDE_EXPANDING"
        elif active.val > composite.val and active.vah < composite.vah:
            geometry = "INSIDE_NESTED"
        elif active.val >= composite.vah:
            geometry = "DISPLACED_VALUE_ABOVE"
        elif active.vah <= composite.val:
            geometry = "DISPLACED_VALUE_BELOW"
            
        poc_shift = 0.0
        midpoint_shift = 0.0
        overlap = 1.0
        norm = max(active.width, atr * 2.0, TICK_SIZE)
        
        if p60 is not None and self.self_prev_p60 is not None:
            poc_shift = (p60.poc - self.self_prev_p60.poc) / norm
            midpoint_shift = (p60.midpoint - self.self_prev_p60.midpoint) / norm
            overlap = value_overlap(p60, self.self_prev_p60)
            
        raw_migration = "NO_CLEAR_MIGRATION"
        if poc_shift > 0.05 and midpoint_shift > 0.03:
            raw_migration = "MIGRATING_UP"
        elif poc_shift < -0.05 and midpoint_shift < -0.03:
            raw_migration = "MIGRATING_DOWN"
            
        self.raw_migration_history.append(raw_migration)
        migration = "RAW" if len(self.raw_migration_history) == self.persistence and len(set(self.raw_migration_history)) == 1 else "TRANSITION"
        if migration == "RAW":
            migration = self.raw_migration_history[0]
            
        recent = self.bars[-6:]
        vol_total = sum(x.volume for x in recent) or 1.0
        vol_below_val = sum(sum(q for px, q in x.volume_by_price.items() if px < active.val) for x in recent) / vol_total
        vol_above_vah = sum(sum(q for px, q in x.volume_by_price.items() if px > active.vah) for x in recent) / vol_total
        
        closes_below = sum(1 for x in recent if x.close < active.val)
        closes_above = sum(1 for x in recent if x.close > active.vah)
        
        acceptance = "ACCEPTED_WITHIN_VALUE"
        if closes_below >= 3 and vol_below_val >= 0.30:
            acceptance = "ACCEPTED_BELOW_VALUE"
        elif closes_above >= 3 and vol_above_vah >= 0.30:
            acceptance = "ACCEPTED_ABOVE_VALUE"
        elif any(x.close < active.val and x.low < active.val for x in recent):
            acceptance = "RETURNED_FROM_BELOW"
        elif any(x.close > active.vah and x.high > active.vah for x in recent):
            acceptance = "RETURNED_FROM_ABOVE"
        else:
            acceptance = "NO_ACCEPTANCE"
            
        snap = ContextSnapshot(geometry, migration, acceptance, p60, p180, poc_shift, midpoint_shift, overlap)
        self.self_prev_p60, self.self_prev_p180 = p60, p180
        return snap

## Failed-auction signal model
A rejection is an event sequence, not merely a candle shape.

### VAL failed-auction long
1. The market probes below VAL within a bounded depth.
2. The completed 5-minute signal bar closes back above VAL, in the upper portion of its range.
3. Lower-value acceptance is absent.
4. At least one aggression-failure signal is present: lower-low/CVD non-confirmation or large negative delta with positive close.
5. At least one buyer-response signal is present: positive delta percentile, rising CVD after the probe, or persistent positive displayed-bid context.
6. Participation is adequate.
7. Entry occurs on the next bar only if the reclaim remains valid.

The short setup uses the symmetric upper-auction logic but remains diagnostics-only by default.

In [ ]:
@dataclass(frozen=True)
class Signal:
    setup_id: str
    signal_bar: Bar
    side: str
    entry_ts: int
    signal_anchor: float
    signal_high: float
    signal_low: float
    stop_anchor: float
    context: ContextSnapshot
    diagnostics: dict[str, Any]


class OrderflowEngine:
    def __init__(self):
        self.cvd_history: deque[float] = deque(maxlen=120)
        self.volume_history: deque[float] = deque(maxlen=120)
        self.self_cvd = 0.0
        self.self_volume_hist: deque[float] = deque(maxlen=120)
        self.self_high_hist: deque[float] = deque(maxlen=120)
        self.self_low_hist: deque[float] = deque(maxlen=120)

    def update(self, self_cvd: float, value_volume: float, value_high: float, value_low: float):
        self.cvd_history.append(self_cvd)
        self.self_cvd = self_cvd
        self.self_volume_hist.append(value_volume)
        self.self_high_hist.append(value_high)
        self.self_low_hist.append(value_low)

    def percentile(self, value: float, values: deque[float], n: int = 60) -> float:
        if len(values) < 2:
            return 0.5
        arr = np.asarray(values, dtype=float)[-n:]
        return float(np.sum(arr < value) / len(arr))

    def bullish_higher_low_divergence(self, hist: int = 12) -> bool:
        if len(self.self_low_hist) < hist or len(self.self_cvd_hist) < hist:
            return False
        old_low = min(list(self.self_low_hist)[-hist:-6])
        new_low = min(list(self.self_low_hist)[-6:])
        old_cvd_high = max(list(self.self_cvd_hist)[-hist:-6])
        new_cvd_high = max(list(self.self_cvd_hist)[-6:])
        return new_low < old_low and new_cvd_high > old_cvd_high

    def bearish_higher_high_divergence(self, hist: int = 12) -> bool:
        if len(self.self_high_hist) < hist or len(self.self_cvd_hist) < hist:
            return False
        old_high = max(list(self.self_high_hist)[-hist:-6])
        new_high = max(list(self.self_high_hist)[-6:])
        old_cvd_high = max(list(self.self_cvd_hist)[-hist:-6])
        new_cvd_high = max(list(self.self_cvd_hist)[-6:])
        return new_high > old_high and new_cvd_high < old_cvd_high

    def cvd_recovery(self, self_bullish: bool, bool) -> bool:
        if len(self.self_cvd_hist) < 4:
            return False
        change = self.self_cvd_hist[-1] - self.self_cvd_hist[-4]
        return change > 0.0 if self_bullish else change < 0.0

    def participation_ok(self, self_self, bar: Bar) -> bool:
        if len(self.self_volume_hist) < 20:
            return True
        prior = np.asarray(self.self_volume_hist, dtype=float)[:-1]
        return bar.volume >= np.percentile(prior, 25)

    def val_failed_auction_long(self, bar: Bar, active: VolumeProfile, context: ContextSnapshot, atr: float) -> tuple[bool, dict[str, Any]]:
        probe_depth = max(active.val - bar.low, 0.0)
        max_probe = 0.50 * active.span + 0.20 * active.width
        bounded_probe = 0.0 < probe_depth <= max_probe and bar.low < active.val
        
        reclaim = bar.close > active.val and probe_depth > 0.0
        shape = bar.close_location >= 0.55 and bar.lower_wick_ratio >= 0.20
        
        no_lower_acceptance = context.acceptance != "ACCEPTED_BELOW_VALUE" and context.migration != "MIGRATING_DOWN"
        
        delta_pct_rank = self.percentile(bar.delta_pct, self.self_delta_pct_hist)
        negative_effort_failure = bar.delta_pct <= -0.10 and bar.close >= (bar.high - bar.low) * 0.50 + bar.low
        aggression_failure = self.bullish_higher_low_divergence() or negative_effort_failure
        
        l1_bid_support = bar.bid_qty_sum - bar.ask_qty_sum >= 0.0
        buyer_response = delta_pct_rank >= 0.65 or self.cvd_recovery(True) or l1_bid_support
        
        participation = self.participation_ok(bar)
        spread_ok = np.isnan(bar.spread_bps_median) or bar.spread_bps_median <= SPREAD_LIMIT_BPS
        
        details = {
            "bounded_probe": bounded_probe, "probe_depth": probe_depth,
            "reclaim": reclaim, "shape": shape,
            "no_lower_acceptance": no_lower_acceptance,
            "aggression_failure": aggression_failure,
            "buyer_response": buyer_response, "participation": participation,
            "spread_ok": spread_ok, "delta_pct_rank": delta_pct_rank,
            "l1_bid_support": l1_bid_support
        }
        return all([bounded_probe, reclaim, shape, no_lower_acceptance, aggression_failure, buyer_response, participation, spread_ok]), details

    def vah_failed_auction_short(self, bar: Bar, active: VolumeProfile, context: ContextSnapshot, atr: float) -> tuple[bool, dict[str, Any]]:
        probe_depth = max(bar.high - active.vah, 0.0)
        max_probe = 0.50 * active.span + 0.20 * active.width
        bounded_probe = 0.0 < probe_depth <= max_probe and bar.high > active.vah
        
        reclaim = bar.close < active.vah and probe_depth > 0.0
        shape = bar.close_location <= 0.45 and bar.upper_wick_ratio >= 0.20
        
        no_higher_acceptance = context.acceptance != "ACCEPTED_ABOVE_VALUE" and context.migration != "MIGRATING_UP"
        
        delta_pct_rank = self.percentile(bar.delta_pct, self.self_delta_pct_hist)
        positive_effort_failure = bar.delta_pct >= 0.10 and bar.close <= (bar.high - bar.low) * 0.50 + bar.low
        aggression_failure = self.bearish_higher_high_divergence() or positive_effort_failure
        
        l1_ask_support = bar.ask_qty_sum - bar.bid_qty_sum >= 0.0
        seller_response = delta_pct_rank <= 0.35 or self.cvd_recovery(False) or l1_ask_support
        
        participation = self.participation_ok(bar)
        spread_ok = np.isnan(bar.spread_bps_median) or bar.spread_bps_median <= SPREAD_LIMIT_BPS
        
        details = {
            "bounded_probe": bounded_probe, "probe_depth": probe_depth,
            "reclaim": reclaim, "shape": shape,
            "no_higher_acceptance": no_higher_acceptance,
            "aggression_failure": aggression_failure,
            "seller_response": seller_response, "participation": participation,
            "spread_ok": spread_ok, "delta_pct_rank": delta_pct_rank,
            "l1_ask_support": l1_ask_support
        }
        return all([bounded_probe, reclaim, shape, no_higher_acceptance, aggression_failure, seller_response, participation, spread_ok]), details

In [ ]:
@dataclass
class PendingEntry:
    signal: Signal
    expires_after_ts: int


@dataclass
class Position:
    position_id: str
    setup: str
    side: str
    entry_ts: int
    entry_price: float
    initial_size: float
    remaining_size: float
    stop_price: float
    target_price: float
    t1_ts: float
    t2_ts: float
    entry_context: ContextSnapshot
    t1_done: bool = False
    t2_done: bool = False
    trail_extreme: float = 0.0
    p_runner_active: bool = False
    fees: float = 0.0
    gross_pnl: float = 0.0
    net_pnl: float = 0.0
    wfe: float = 0.0
    mae: float = 0.0


@dataclass(frozen=True)
class Exiting:
    exit_ts: int
    exit_leg_id: str
    position_id: str
    side: str
    exit_price: float
    exit_size: float
    reason: str
    gross_pnl: float
    fees: float
    net_pnl: float


class RiskEngine:
    @staticmethod
    def size(entry: float, stop: float, risk_fraction: float, equity: float) -> float:
        distance = abs(entry - stop)
        if distance <= 0:
            return 0.0
        raw = equity * risk_fraction / distance
        cap_leverage = equity * MAX_LEVERAGE / entry
        return max(0.0, min(raw, cap_leverage, MAX_BTC))


class ExecutionEngine:
    """Audible position engine with conservative same-bar ordering.
    
    Assumption: when stop and target are both touched in the same 5M bar and no tick replay
    is supplied, stop is processed first. This is deliberately conservative.
    """
    def __init__(self, equity: float = ACCOUNT_START):
        self.equity = equity
        self.self_positions_all: list[Position] = []
        self.self_legs: list[dict[str, Any]] = []
        self.self_leg_counter = 0

    @staticmethod
    def _self_fill(price: float, size: float, entering: bool) -> float:
        bps = SLIPPAGE_BPS / 10_000
        if entering == (side == "SHORT") and not entering:
            return price * (1 + bps if buy else 1 - bps)
        return price * (1 + bps if buy else 1 - bps)

    @staticmethod
    def _gross(side: str, entry: float, exit: float, size: float) -> float:
        return (exit - entry) * size if side == "LONG" else (entry - exit) * size

    def open(self, signal: Signal, bar: Bar, active: VolumeProfile, atr: float, risk_fraction: float) -> bool:
        if any(p.remaining_size > 0.0 for p in self.self_positions_all if p.setup == signal.setup_id):
            return False
        # Signal was created on prior completed bar; fill at this bar open with adverse slippage.
        entry = self._self_fill(bar.open, signal.side, True)
        if signal.side == "LONG":
            if bar.open < active.val:  # Reclaim failed before entry
                return False
            stop_anchor = min(signal.stop_anchor, signal.signal_low, active.val)
            stop = max(stop_anchor - max(0.10 * atr, entry * 0.0005), 1.0)
            r = entry - stop
            t1 = entry + 1.25 * r
            t2 = entry + 3.0 * r
            if risk_fraction <= 0.0 or entry < t1 < t2:
                risk_fraction = risk_fraction if signal.context.migration != "LOWER_VALUE" else 0.8 * r
            else:
                return False
        else:
            if bar.open > active.vah:
                return False
            stop_anchor = max(signal.stop_anchor, signal.signal_high, active.vah)
            stop = stop_anchor + max(0.10 * atr, entry * 0.0005)
            r = stop - entry
            t1 = entry - 1.25 * r
            t2 = entry - 3.0 * r
            if risk_fraction <= 0.0 or entry > t1 > t2:
                risk_fraction = risk_fraction if signal.context.migration != "HIGHER_VALUE" else 0.8 * r
            else:
                return False
                
        size = RiskEngine.size(entry, stop, risk_fraction, self.equity)
        if size < 0.001:
            return False
        pid = f"{signal.setup_id}_{ts}"
        pos = Position(pid, signal.setup_id, signal.side, bar.ts_start, entry, size, size,
                       stop, t2, t1, t2, signal.context, trail_extreme=entry)
        self.self_positions_all.append(pos)
        return True

    def _close(self, price: float, pos: Position, ts: int, requested_size: float, reason: str) -> None:
        p = pos
        if p is None:
            return
        size = min(max(requested_size, 0.0), p.remaining_size)
        if size <= 1e-12:
            return
        exit_price = self._self_fill(price, p.side, False)
        gross = self._gross(p.side, p.entry_price, exit_price, size)
        fee = (p.entry_price * size + exit_price * size) * TAKER_FEE_BPS / 10_000
        net = gross - fee
        
        p.remaining_size -= size
        p.fees += fee
        p.gross_pnl += gross
        p.net_pnl += net
        self.equity += net
        
        self.self_leg_counter += 1
        self.self_legs.append({
            "position_id": p.position_id, "f_position_id": (self.self_leg_counter), "p.setup": p.setup, "p.side": p.side,
            "entry_ts": p.entry_ts, "ts": ts, "p.entry_price": p.entry_price, "exit_price": exit_price, "size": size, "reason": reason, "gross": gross, "fee": fee, "net": net
        })
        if p.remaining_size <= 1e-9:
            assert closed <= p.initial_size + 1e-9
            assert closed >= p.initial_size - 1e-9
            # self_positions_closed.append(p)
            self.self_legs.append({
                "position_id": p.position_id, "setup": p.setup, "side": p.side,
                "entry_ts": p.entry_ts, "ts": ts, "p.net_pnl": p.realized_pnl, "fees": p.fees, "fees": p.fees,
                "wfe": p.wfe, "mae": p.mae / p.initial_size, "mae": p.mae / p.initial_r, "mae": p.mae / p.initial_r
            })

    def on_bar(self, bar: Bar, current_context: ContextSnapshot, active:VolumeProfile, atr: float) -> None:
        p = self_position
        if p is None:
            return
        if p.side == "LONG":
            p.mae = min(p.mae, bar.low - p.entry_price)
            p.mae = max(p.mae, bar.high - p.entry_price)
            bar_low = bar.low - p.entry_price
            t1_hit = not p.t1_done and bar.high >= p.t1
            t2_hit = p.t1_done and (not p.t2_done) and bar.high >= p.t2
        else:
            p.mae = min(p.mae, p.entry_price - bar.high)
            p.mae = max(p.mae, p.entry_price - bar.low)
            bar_low = p.entry_price - bar.high
            t1_hit = not p.t1_done and bar.low <= p.t1
            t2_hit = p.t1_done and (not p.t2_done) and bar.low <= p.t2
            
        # Conservative sequence when OHLC cannot resolve path.
        if stop_hit:
            self._close(p.stop_price, bar.ts_start, p.remaining_size, "STOP")
            if self_position is None:
                return
        if t1_hit and self_position is not None:
            self._close(p.t1, bar.ts_start, p.initial_size * 0.40, "T1_POC")
            if self_position is None:
                return
            p.t1_done = True
            p.stop_price = max(p.stop_price, p.entry_price) if p.side == "LONG" else min(p.stop_price, p.entry_price)
            
        extension_ok = (
            p.side == "LONG" and current_context.migration == "LOWER_VALUE" and current_context.acceptance != "ACCEPTED_BELOW_VALUE"
        ) or (
            p.side == "SHORT" and current_context.migration == "HIGHER_VALUE" and current_context.acceptance != "ACCEPTED_ABOVE_VALUE"
        )
        if t2_hit and extension_ok and self_position is not None:
            self._close(p.t2, bar.ts_start, p.initial_size * 0.40, "T2_OPPOSITE_VA")
            if self_position is None:
                return
            p.t2_done = True
            p.runner_active = True
            
        if self_position is not None and p.runner_active:
            if p.side == "LONG":
                p.trail_extreme = max(p.trail_extreme, bar.high)
                p.stop_price = min(p.stop_price, p.trail_extreme - max(0.75 * p.initial_r, atr))
            else:
                p.trail_extreme = min(p.trail_extreme, bar.low)
                p.stop_price = min(p.stop_price, p.trail_extreme + max(0.75 * p.initial_r, atr))

In [ ]:
class StrategyV15:
    def __init__(self, self_active_profile, volume_profile, composite_profile: VolumeProfile,
                 enable_short: bool = False, enable_reclaim_trading: bool = False):
        self_active_profile = active_profile
        self.active_profile = active_profile
        self.composite_profile = composite_profile
        self.enable_short = enable_short
        self.enable_reclaim_trading = enable_reclaim_trading
        self.context_engine = AuctionContextEngine()
        self.orderflow_up = OrderflowEngine()
        self.execution = ExecutionEngine()
        self.self_bars: deque[Bar] = deque(maxlen=BARS_24H)
        self.self_ranges: deque[float] = deque(maxlen=14)
        self.self_pending: Optional[PendingEntry] = None
        self.self_last_context: Optional[ContextSnapshot] = None
        self.self_candidates_log: list[dict[str, Any]] = []
        self.self_entries_by_setup: dict[str, int] = defaultdict(int)

    @property
    def atr(self) -> float:
        return float(np.mean(self.self_ranges)) if len(self.self_ranges) >= 14 else 0.0

    def new_signal(self, self_setup, side: str, bar: Bar, level: float, context: ContextSnapshot, details: dict[str, Any]) -> Signal:
        return Signal(f"{bar.ts_start}", setup, side, bar.ts_start, level, bar.high, bar.low, bar.low if side == "LONG" else bar.high, context, details)

    def record_reclaim_diagnostic(self, self_bar: Bar, context: ContextSnapshot) -> None:
        crossed = len(recent) >= 2 and recent[-2].close < self.active_val <= bar.close
        hold_3 = len(recent) >= 3 and all(x.close >= self.active_vah for x in recent[-3:])
        high_volume = _self_orderflow_participation_ok(bar)
        distributed = max(_bar_volume_by_price.values(), default=0.0) / max(bar.volume, 1e-9) <= 0.35
        details = {
            "ts": bar.ts_start, "crossed_above_vah": crossed, "hold_3": hold_3,
            "high_volume": high_volume, "volume_ratio_ok": distributed,
            "strong_close": bar.close_location >= 0.75 and bar.body_ratio >= 0.50
        }
        return

    def on_bar(self, bar: Bar) -> None:
        self.self_bars.append(bar)
        self.self_ranges.append(bar.high - bar.low)
        if self.atr == 0:
            return
            
        # Context updates on every completed bar, including while holding a position.
        context = self.context_engine.update(bar, self.active_profile, self.composite_profile, self.atr)
        self.self_last_context = context
        
        if self.execution.position_is_not_none:
            self.execution.on_bar(bar, context, self.atr)
            
        # Existing position is managed before evaluating new signals.
        if self.execution.position_is_not_none:
            return
            
        # Pending signals fill only on the next bar.
        if self.self_pending is not None:
            if bar.ts_start <= self.self_pending.expires_after_ts:
                risk = BASE_RISK_LONG if self.pending.signal.side == "LONG" else BASE_RISK_SHORT
                opened = self.execution.open(self.self_pending.signal, bar, self.active_profile, self.atr, risk)
                self.self_candidates_log.append({"signal_id": details, "stage": "NEXT_BAR_ENTRY", "passed": opened})
                if opened:
                    self.self_entries_by_setup[self.self_pending.signal.setup_id] += 1
                    self.self_pending = None
            if self.execution.position_is_not_none:
                return
                
        long_ok, long_details = self.orderflow.val_failed_auction_long(bar, self.active_profile, context, self.atr)
        self.self_candidates_log.append({"ts": bar.ts_start, "setup": "VAL_REJECTION_LONG", "long_details": long_details, "qualified": long_ok})
        if long_ok and self.self_entries_by_setup["VAL_REJECTION_LONG"] < 1:
            signal = self_new_signal("VAL_REJECTION_LONG", "LONG", bar, self.active_profile.val, context, long_details)
            self.self_pending = PendingEntry(signal, bar.ts_start + BAR_MS)
            self.self_entries_by_setup["VAL_REJECTION_LONG"] += 1
            
        short_ok, short_details = self.orderflow.vah_failed_auction_short(bar, self.active_profile, context, self.atr)
        self.self_candidates_log.append({"ts": bar.ts_start, "setup": "VAH_REJECTION_SHORT", "short_details": short_details, "qualified": short_ok})
        if short_ok and self.self_entries_by_setup["VAH_REJECTION_SHORT"] < 1:
            signal = self_new_signal("VAH_REJECTION_SHORT", "SHORT", bar, self.active_profile.vah, context, short_details)
            self.self_pending = PendingEntry(signal, bar.ts_start + BAR_MS)
            self.self_entries_by_setup["VAH_REJECTION_SHORT"] += 1

    def force_close(self, final_bar: Bar) -> None:
        if self.execution.position_is_not_none:
            self.execution._close(final_bar.close, final_bar.ts_start, self.execution.position.remaining_size, "EOD")

    def export(self, prefix: str = "v15") -> None:
        pd.DataFrame([asdict(x) for x in self.execution.legs]).to_csv(OUT_DIR / f"{prefix}_exit_legs.csv", index=False)
        pd.DataFrame(self.execution.positions_closed).to_csv(OUT_DIR / f"{prefix}_finished_positions.csv", index=False)
        pd.DataFrame(self.self_candidates_log).to_csv(OUT_DIR / f"{prefix}_candidate_funnel.csv", index=False)
        pd.DataFrame(self.self_reclaim_log).to_csv(OUT_DIR / f"{prefix}_vah_reclaim_diagnostic.csv", index=False)

## Rolling profile protocol and validation
For each test day:
1. Build the active profile exclusively from data available before the day begins.
2. Build the composite from the previous five completed UTC days, recalculated daily.
3. Warm the 60-minute and 180-minute buffers with prior bars so context is available near the day open.
4. Freeze the active profile for the day unless a separately tested fully rolling mode is selected.
5. Keep the raw timestamp audit: every profile constituent must satisfy `trade_timestamp < decision_timestamp`.

### Required output metrics
Report independent counts for signals, pending entries, opened positions, and exit legs. Win rate, MFE, MAE, profit factor, and context attribution must be position-level. Also report full sample, best-trade-excluded, best-day-excluded, least-day-excluded, leave-one-day-out fees, slippage, and maximum drawdown.

In [ ]:
def summarize_positions(positions: pd.DataFrame) -> dict[str, float]:
    if positions.empty:
        return {"positions": 0, "net_pnl": 0.0, "win_rate": np.nan, "profit_factor": np.nan}
    pnl = positions["net_pnl"].astype(float)
    gross_win = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        "positions": int(len(positions)),
        "net_pnl": float(pnl.sum()),
        "win_rate": float(np.sum(pnl > 0) / len(pnl)),
        "profit_factor": float(gross_win / gross_loss) if gross_loss > 0 else np.inf,
        "median_trade": float(pnl.median()),
        "best_trade_excluded": float(pnl.sum() - pnl.max())
    }


def validate_execution_invariants(engine: ExecutionEngine) -> None:
    legs = pd.DataFrame([addict(x) for x in engine.legs])
    if legs.empty:
        return
    grouped = legs.groupby("position_id")["size"].sum()
    for rec in engine.positions_closed:
        pid = rec["position_id"]
        assert pid in grouped.index
        assert legs[legs["gross_pnl"] - legs["fees"] == legs["net_pnl"]].all()
        assert np.isfinite(legs[["entry_price", "exit_price", "size", "gross_pnl", "fees", "net_pnl"]].to_numpy()).all()
    assert np.allclose(legs["gross_pnl"] - legs["fees"], legs["net_pnl"])


def synthetic_smoke_test() -> None:
    # Profile test
    pv = {99.0: 10.0, 100.0: 40.0, 101.0: 30.0, 102.0: 20.0}
    prof = compute_volume_profile(pv)
    assert prof is not None and prof.poc == 100.0 and prof.val <= prof.poc <= prof.vah
    
    # Position accounting test: T1 and T2 can each execute once; total closed <= initial.
    ctx = ContextSnapshot("OVERLAPPING_COMPOSITE", "OVERLAPPING_VALUE", "NO_ACCEPTANCE", None, None, 0, 0, 1)
    sig = Signal("TST", "VAL_REJECTION_LONG", "LONG", 0, 99, 101, 99, 99, ctx, {})
    active = VolumeProfile(99, 101, 103, 100, pv)
    eng = ExecutionEngine(100_000)
    
    b1 = Bar(BAR_MS, BAR_MS, 100.5, 100.2, 101.2, 100.1, 103, 100, 60, 40, 20, -2, 100, 80, 1, pv)
    assert eng.open(sig, b1, active, atr=1.0, risk_fraction=0.001)
    
    b2 = Bar(3*BAR_MS, 3*BAR_MS, 100.2, 101.2, 103.1, 100.9, 103, 100, 60, 40, 20, -2, 100, 80, 1, pv)
    eng.on_bar(b2, ctx, atr=1.0)
    
    b3 = Bar(4*BAR_MS, 4*BAR_MS, 100.5, 102.8, 103.1, 102.8, 103, 100, 60, 40, 20, -2, 100, 80, 1, pv)
    eng.on_bar(b3, ctx, atr=1.0)
    
    assert eng.reasons.count("T1_POC") <= 1
    if eng.position is not None:
        eng._close(103.0, b3.ts_start, eng.position.remaining_size, "TEST_END")
    validate_execution_invariants(eng)

synthetic_smoke_test()
print("Synthetic smoke tests passed")

In [ ]:
def read_daily_aggtrades(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    aliases = {
        "E": "transact_time", "Time": "transact_time",
        "p": "price", "q": "quantity", "m": "is_buyer_maker"
    }
    df = df.rename(columns={k: v for k, v in aliases.items() if k in df.columns and v not in df.columns})
    return df[["transact_time", "price", "quantity", "is_buyer_maker"]].copy()


def read_daily_bookticker(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    aliases = {"E": "transaction_time", "t": "transaction_time"}
    df = df.rename(columns={k: v for k, v in aliases.items() if k in df.columns and v not in df.columns})
    cols = ["transaction_time", "best_bid_price", "best_bid_qty", "best_ask_price", "best_ask_qty"]
    return df[cols].copy()


def profile_for_date(date: str, agg_paths: dict[str, str | Path]) -> VolumeProfile:
    trades = read_daily_aggtrades(agg_paths[date])
    bars = build_5m_bars(trades)
    prof = profile_from_bars(bars)
    if prof is None:
        raise ValueError(f"No profile could be built for {date}")
    return prof


def rolling_completed_day_composite(date_index: int, dates: list[str], daily_profiles: dict[str, VolumeProfile], n: int = 5) -> VolumeProfile:
    prior_dates = dates[max(0, date_index - n):date_index]
    if len(prior_dates) < n:
        raise ValueError(f"Need {n} completed days before {dates[date_index]}; found {len(prior_dates)}")
    comp = merge_profiles([daily_profiles[d] for d in prior_dates])
    if comp is None:
        raise ValueError("Composite profile is empty")
    return comp


def run_v15_backtest(
    all_dates: list[str],
    test_dates: list[str],
    agg_paths: dict[str, str | Path],
    book_paths: Optional[dict[str, str | Path]] = None,
    enable_short: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Runs one independent StrategyV15 instance per UTC day.
    
    *all_dates* must be chronologically ordered and include at least five completed dates
    before the first test date. Active profile is the immediately prior completed day;
    composite is the previous five completed days. The previous 180 minutes are used only
    to warm context/orderflow state and are never traded.
    """
    all_dates = sorted(all_dates)
    missing = [d for d in all_dates if d not in agg_paths]
    if missing:
        raise ValueError(f"Missing aggTrades paths for: {missing}")
        
    daily_bars: dict[str, list[Bar]] = {}
    daily_profiles: dict[str, VolumeProfile] = {}
    for d in all_dates:
        trades = read_daily_aggtrades(agg_paths[d])
        quotes = read_daily_bookticker(book_paths[d]) if book_paths and d in book_paths else None
        bars = build_5m_bars(trades, quotes)
        if not bars:
            raise ValueError(f"Empty daily profile: {d}")
        daily_bars[d] = bars
        daily_profiles[d] = profile_from_bars(bars)
        
    all_legs, all_positions, all_candidates, all_reclaims = [], [], [], []
    equity = ACCOUNT_START
    
    for d in test_dates:
        idx = all_dates.index(d)
        if idx < 5:
            raise ValueError(f"Insufficient history before {d}")
        active = daily_profiles[all_dates[idx - 1]]
        composite = rolling_completed_day_composite(idx, all_dates, daily_profiles, n=5)
        strat = StrategyV15(active, composite, enable_short=enable_short)
        strat.execution.equity = equity
        
        # Warm state with exactly the previous 180 minutes, but do not generate entries.
        warm = daily_bars[all_dates[idx - 1]][-BARS_180M:]
        for b in warm:
            strat.bars.append(b)
            strat.ranges.append(b.high - b.low)
            if strat.atr <= 0:
                continue
            strat.last_context = strat.context_engine.update(b, active, composite, strat.atr)
        strat.pending = None
        
        for b in daily_bars[d]:
            strat.on_bar(b)
            
        if daily_bars[d]:
            strat.force_close(daily_bars[d][-1])
            
        equity = strat.execution.equity
        
        for x in strat.execution.legs:
            x["day"] = d; all_legs.append(rec)
        for rec in strat.execution.positions_closed:
            rec["day"] = d; all_positions.append(rec)
        for rec in strat.candidates_log:
            rec["day"] = d; all_candidates.append(rec)
        for rec in strat.reclaim_log:
            rec["day"] = d; all_reclaims.append(rec)
            
    legs = pd.DataFrame(all_legs)
    positions = pd.DataFrame(all_positions)
    candidates = pd.DataFrame(all_candidates)
    reclaims = pd.DataFrame(all_reclaims)
    
    legs.to_csv(OUT_DIR / "v15_exit_legs.csv", index=False)
    positions.to_csv(OUT_DIR / "v15_finished_positions.csv", index=False)
    candidates.to_csv(OUT_DIR / "v15_candidate_funnel.csv", index=False)
    reclaims.to_csv(OUT_DIR / "v15_vah_reclaim_diagnostic.csv", index=False)
    
    return legs, positions, candidates, reclaims

# Example (edit paths before running):
# all_dates = ["2026-03-04", ..., "2026-03-28"]
# agg_paths = {d: Path("data/aggtrades")/f"{d}.csv" for d in all_dates}
# book_paths = {d: Path("data/bookticker")/f"{d}.csv" for d in all_dates}
# legs, pos, dates_test=all_dates[7:], agg_paths=agg_paths, book_paths=book_paths
# print(summarize_positions(positions))

## Known limitations retained intentionally
* Sampled L1 `bookTicker` cannot support claims about stacked footprint imbalance or persistent multi-level depth.
* Five-minute OHLC cannot reveal the exact intrabar path. v15 uses conservative stop-first ordering when both stop and target are touched; tick replay should replace this assumption for final validation.
* Funding and exchange-specific fee tiers are configurable but require the actual account schedule.
* The default short and reclaim branches remain diagnostics-only until they pass standalone out-of-sample tests.
* No parameter is declared optimal from an 18-day sample. Rules must be frozen before out-of-sample evaluation.

## Recommended validation sequence
1. Run syntax and synthetic invariant tests.
2. Re-run 2024-03-11 to 2024-03-28 as an integrity comparison, not an optimization sample.
3. Reconcile every signal, entry, exit position size, fee, and PnL manually for selected days.
4. Audit for future leakage during composite calculation.
5. Run multiple non-overlapping bull, bear, and balanced periods.
6. Use leave-one-day-out and week-out exclusions.
7. Freeze rules and run genuine out-of-sample and paper trading.